# Colab: ck500 hạt giống 202 và nhánh kiểm "xáo thưởng"

Mô tả đầy đủ, cách đọc kết quả và bước chấm test: `harness/runbook/colab_ck500_s202_va_kiem_thuong.md`.

**Thứ tự:** chạy ô 0, 1, 2, 3, 4 (thăm dò), rồi 5 và 6 ngay sau nhau. Ô 6 chạy foreground tới hết lượt, **không bấm Stop**. Xong thì ô 7.
**Mất máy:** mở lại, chạy ô 0, 1, 2, 3, rồi 5 và 6 (bỏ ô 4). Ô 2 tự kéo điểm lưu từ Drive về.
**Hai lượt:** đổi `RUN` ở ô 0 (`"s202"` rồi `"xao101"`), mỗi lượt một runtime mới.

In [ ]:
# Ô 0 — cấu hình
RUN = "s202"        # "s202" = ck500 hạt 202 · "xao101" = xáo thưởng, hạt 101 (đúng 1.000 câu nhắc của ck500)
assert RUN in ("s202", "xao101")
D = "/content/drive/MyDrive/thesis/ck500_s202"     # Drive: fgrb_p1_bundle.zip + grpo_spice.py + build_branch_data.py
W = "/content"
RUNARGS = {"s202": ["--seed", "202"], "xao101": ["--reward", "shuffle"]}[RUN]
MD5_GRPO = "8489108147c8d1ed80c3dc67c8ad3d3f"
MD5_BBD = "619e63e123a6dbf60086e65ee94a3912"
import time, os; T_NB = time.time()
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
print("RUN =", RUN, "·", RUNARGS)

In [ ]:
# Ô 1 — GPU, gói, Java 8
import subprocess, sys, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode

g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
print("GPU:", g.stdout.strip() or "⛔ KHÔNG THẤY GPU (Runtime → Change runtime type)")
assert g.returncode == 0 and g.stdout.strip(), "DỪNG: không có GPU"
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java")
assert sh("java -version") == 0, "DỪNG: không có Java"
sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate bitsandbytes pycocoevalcap 'trl==0.29.1' datasets pillow 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -2")      # Colab có torchao 0.10 cũ, peft 0.21 từ chối (bẫy đã gặp 6/10 và 23/10)
r = subprocess.run([sys.executable, "-c",
    "import trl, transformers, peft, torch; "
    "print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.version.cuda, "
    "torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))"], capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
assert r.returncode == 0, "DỪNG: import lỗi. Nếu lỗi nhắc torchao: !pip uninstall -y torchao rồi chạy lại ô này"
assert r.stdout.split()[0] == "0.29.1", "DỪNG: trl không đúng bản pin 0.29.1"
assert "None" not in r.stdout, "DỪNG: torch bản CPU, cài lại torch bản CUDA rồi Restart runtime"
cap = int(r.stdout.strip().split("(")[-1].split(",")[0])
print("dtype sẽ dùng:", "bfloat16" if cap >= 8 else "float16",
      "(ck500 gốc chạy float16 trên Kaggle T4; L4/A100 chạy bfloat16 ⇒ phải khai khi báo)")

In [ ]:
# Ô 2 — Drive, bung gói, kiểm md5, kéo điểm lưu cũ (nếu có)
from google.colab import drive
drive.mount("/content/drive")
import os, glob, shutil, hashlib, zipfile, re
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()

assert os.path.isdir(D), f"DỪNG: chưa có thư mục {D} trên Drive"
for f in ("fgrb_p1_bundle.zip", "grpo_spice.py", "build_branch_data.py"):
    assert os.path.exists(f"{D}/{f}") or (f.endswith(".zip") and os.path.isdir(f"{W}/fgrb_p1_bundle")), f"DỪNG: thiếu {D}/{f}"

if not os.path.isdir(f"{W}/fgrb_p1_bundle"):
    shutil.copy(f"{D}/fgrb_p1_bundle.zip", f"{W}/fgrb_p1_bundle.zip")
    zipfile.ZipFile(f"{W}/fgrb_p1_bundle.zip").extractall(W)
    os.remove(f"{W}/fgrb_p1_bundle.zip")
BUNDLE = f"{W}/fgrb_p1_bundle"
assert os.path.isdir(f"{BUNDLE}/adapter_s1_seed101") and os.path.isdir(f"{BUNDLE}/images"), "DỪNG: gói thiếu thư mục"
for f in ("p1_train_rows.jsonl", "p1_val_rows.jsonl", "ocr.jsonl"):
    assert os.path.exists(f"{BUNDLE}/{f}"), f"DỪNG: gói thiếu {f}"
print("số ảnh:", len(os.listdir(f"{BUNDLE}/images")))

for f in ("grpo_spice.py", "build_branch_data.py"):
    shutil.copy(f"{D}/{f}", f"{W}/{f}")
assert md5(f"{W}/grpo_spice.py") == MD5_GRPO, "DỪNG: grpo_spice.py trên Drive là bản cũ"
assert md5(f"{W}/build_branch_data.py") == MD5_BBD, "DỪNG: build_branch_data.py lệch bản máy nhà"

MERGED = f"{W}/s1_merged"
OUT = f"{W}/out_{RUN}"; DRV = f"{D}/{RUN}"
os.makedirs(OUT, exist_ok=True); os.makedirs(DRV, exist_ok=True)

def so_ck(p):
    m = re.fullmatch(r"checkpoint-(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else None      # loại thư mục dở "checkpoint-N.tmp"

for ck in glob.glob(f"{DRV}/checkpoint-*"):
    if so_ck(ck) is None: continue
    dst = f"{OUT}/{os.path.basename(ck)}"
    if not os.path.exists(dst):
        shutil.copytree(ck, dst)
print("RUN", RUN, "· điểm lưu có sẵn:", sorted(so_ck(p) for p in glob.glob(f"{OUT}/checkpoint-*") if so_ck(p) is not None))
print("md5 khớp ✅")

In [ ]:
# Ô 3 — jar Stanford cho SPICE, selftest (0 GPU, có SPICE) và hoà S1
# SPICE cần 2 jar Stanford CoreNLP (385 MB). Colab không tới được nlp.stanford.edu (No route to host) ⇒ lấy từ Drive.
import importlib.util
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
JARS = {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}
for j, sz in JARS.items():
    dst = f"{SPL}/{j}"
    if os.path.exists(dst) and os.path.getsize(dst) == sz: continue
    src = f"{D}/stanford_jars/{j}"
    assert os.path.exists(src), f"DỪNG: thiếu {src} — upload thư mục stanford_jars (2 jar) lên Drive"
    shutil.copy(src, dst)
    assert os.path.getsize(dst) == sz, f"DỪNG: {j} chép lỗi (sai kích thước)"
print("jar SPICE đủ ✅", {j: os.path.getsize(f"{SPL}/{j}") for j in JARS})

def chay(args, tenlog):
    r = subprocess.run(["python", "grpo_spice.py"] + args, cwd=W, capture_output=True, text=True)
    open(f"{W}/{tenlog}", "w").write(r.stdout + r.stderr)
    print((r.stdout + r.stderr)[-1800:], flush=True)
    return r.returncode

assert chay(["--selftest", "--bundle", BUNDLE], "selftest.log") == 0, "DỪNG: selftest lỗi"
if not os.path.exists(f"{MERGED}/config.json"):
    assert chay(["--merge", "--bundle", BUNDLE, "--merged", MERGED], "merge.log") == 0, "DỪNG: hoà S1 lỗi"
assert os.path.exists(f"{MERGED}/config.json"), "DỪNG: chưa có bản hoà"
print("hoà xong ✅")

In [ ]:
# Ô 4 — THĂM DÒ 20 bước (50 câu nhắc dài nhất). Đo s/bước + VRAM, và là phép kiểm mã mới trên GPU. Bỏ ô này khi chạy tiếp sau mất máy.
OUTP = f"{W}/probe_{RUN}"
shutil.rmtree(OUTP, ignore_errors=True)
log = open(f"{W}/probe.log", "w")
p = subprocess.Popen(["python", "grpo_spice.py", "--probe", "--no-q4", "--bs", "4", "--accum", "4", "--gen-chunk", "8"] + RUNARGS
                     + ["--bundle", BUNDLE, "--merged", MERGED, "--out", OUTP],
                     cwd=W, stdout=log, stderr=subprocess.STDOUT, text=True, start_new_session=True, env=ENV)
t0 = time.time()
while p.poll() is None:
    time.sleep(30)
    print(f"{(time.time()-t0)/60:.1f} phút · log {os.path.getsize(f'{W}/probe.log')} byte", flush=True)
L = open(f"{W}/probe.log", encoding="utf-8", errors="ignore").read().splitlines()
for l in L:
    if l.startswith(("[cấu hình]", "[xong]", "[lô]", "GRPOConfig:", "[sinh theo khúc]")): print(l[:300])
print("mã thoát:", p.returncode)
want = {"s202": "[cấu hình] seed 202 · thưởng spice", "xao101": "[cấu hình] seed 101 · thưởng shuffle"}[RUN]
assert p.returncode == 0, "DỪNG: probe lỗi, đọc probe.log"
assert any(l.startswith(want) for l in L), f"DỪNG: không thấy dòng '{want}' (mã là bản cũ?)"
assert any(l.startswith("[xong] 20 bước") for l in L), "DỪNG: probe không chạy đủ 20 bước"
assert not any("Traceback" in l or "OutOfMemory" in l for l in L), "DỪNG: Traceback/OOM"
xong = [l for l in L if l.startswith("[xong]")][-1]
spb = float(re.search(r"([\d.]+) s/bước", xong).group(1))
print(f"\n→ ~{spb:.0f} s/bước ⇒ 500 bước ≈ {500*spb/3600:.1f} h (probe dùng câu nhắc dài nhất nên thực tế hơi thấp hơn)")
print("→ so với đỉnh VRAM ở dòng [xong]: L4 chỉ chọn khi ≤ ~17 GiB. Quyết máy Ở ĐÂY, không đổi giữa chừng.")
shutil.rmtree(OUTP, ignore_errors=True)

In [ ]:
# Ô 5 — TRAIN (chạy nền, trả về ngay) + đồng bộ điểm lưu sang Drive. Chạy ô 6 NGAY SAU.
import threading, ast, statistics as st

def du_tep(ck):
    return all(os.path.exists(f"{ck}/{f}") for f in ("trainer_state.json", "optimizer.pt", "adapter_model.safetensors"))

def kich_thuoc(d):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(d) for f in fs)

def dong_bo():
    while True:
        time.sleep(300)
        try:
            for ck in sorted(glob.glob(f"{OUT}/checkpoint-*")):
                n = so_ck(ck); dst = f"{DRV}/checkpoint-{n}"
                if n is None or os.path.exists(dst) or not du_tep(ck): continue
                if time.time() - os.path.getmtime(f"{ck}/trainer_state.json") < 30: continue   # trainer ghi trainer_state.json CUỐI CÙNG
                shutil.rmtree(dst + ".tmp", ignore_errors=True)
                shutil.copytree(ck, dst + ".tmp")
                if kich_thuoc(dst + ".tmp") != kich_thuoc(ck):
                    shutil.rmtree(dst + ".tmp", ignore_errors=True); continue
                os.rename(dst + ".tmp", dst)
            # giữ Drive gọn: CHỈ 2 điểm mới nhất (xoá hết còn lại, kể cả 250)
            tren = sorted((so_ck(p), p) for p in glob.glob(f"{DRV}/checkpoint-*") if so_ck(p) is not None)
            for n, p_ in tren[:-2]:
                shutil.rmtree(p_, ignore_errors=True)
            shutil.copy(f"{W}/train_{RUN}.log", f"{DRV}/train_{RUN}.log.cp"); os.replace(f"{DRV}/train_{RUN}.log.cp", f"{DRV}/train_{RUN}.log")
        except Exception as e:
            print("⚠️ đồng bộ Drive lỗi:", repr(e), flush=True)
threading.Thread(target=dong_bo, daemon=True).start()

log = open(f"{W}/train_{RUN}.log", "a")
cmd = ["python", "grpo_spice.py", "--train", "--no-q4", "--bs", "4", "--accum", "4", "--gen-chunk", "8"] + RUNARGS \
      + ["--bundle", BUNDLE, "--merged", MERGED, "--out", OUT, "--resume", "auto"]
P = subprocess.Popen(cmd, cwd=W, stdout=log, stderr=subprocess.STDOUT, text=True, start_new_session=True, env=ENV)
T_TRAIN = time.time()
print("PID", P.pid, "·", " ".join(cmd), flush=True)

In [ ]:
# Ô 6 — THEO DÕI foreground tới hết lượt. KHÔNG bấm Stop. (Cần làm việc khác: mở notebook thứ hai.)
TONG = 500

def ck_moi():
    c = sorted(so_ck(p) for p in glob.glob(f"{OUT}/checkpoint-*") if so_ck(p) is not None)
    return c[-1] if c else "chưa có"

def tinh_trang():
    L = open(f"{W}/train_{RUN}.log", encoding="utf-8", errors="ignore").read().splitlines()
    tiep = [l for l in L if l.startswith("[tiếp từ]")]
    goc = int(re.findall(r"checkpoint-(\d+)", tiep[-1])[0]) if tiep and "checkpoint-" in tiep[-1] else 0
    D_ = []
    for l in L:
        if l.startswith("{'loss'"):
            try: D_.append({k: float(v) for k, v in ast.literal_eval(l).items() if re.match(r"^[-\d.e+na]+$", str(v))})
            except Exception: pass
    sp = [l for l in L if l.startswith("[spice]")][-20:]
    nDrv = len([p for p in glob.glob(f"{DRV}/checkpoint-*") if so_ck(p) is not None])
    msg = f"{(time.time()-T_NB)/3600:5.2f} h · bước {goc+len(D_)}/{TONG} · lưu cuối {ck_moi()} · Drive {nDrv} điểm"
    for tag in ("[cấu hình]", "[nạp default]"):
        x = [l for l in L if l.startswith(tag)]
        if x: msg += f"\n      {x[-1][:100]}"
    if D_:
        spb = (time.time() - T_TRAIN) / len(D_)
        kl = [d["kl"] for d in D_[-20:] if "kl" in d]
        th = [float(re.search(r"thưởng TB (-?[\d.]+)", l).group(1)) for l in sp]
        msg += (f"\n      {spb:.0f} s/bước · còn ~{(TONG-goc-len(D_))*spb/3600:.1f} h"
                f"\n      20 bước gần nhất: thưởng TB {st.mean(th) if th else float('nan'):.3f} · "
                f"kl TB {st.mean(kl) if kl else float('nan'):.4f} · grad≈0 {sum(d.get('grad_norm',1)<1e-3 for d in D_[-20:])}/20")
        if any(v != v for d in D_[-5:] for v in d.values()): msg += "\n      ⚠️ nan trong 5 bước cuối"
    if any("Traceback" in l or "OutOfMemory" in l for l in L[-80:]): msg += "\n      ⚠️ Traceback/OOM ở cuối log"
    return msg

while P.poll() is None:
    time.sleep(120)
    print(tinh_trang(), flush=True)
print("TIẾN TRÌNH KẾT THÚC · mã thoát", P.returncode, flush=True)
print(tinh_trang(), flush=True)

In [ ]:
# Ô 7 — XONG: kiểm toàn vẹn, đóng gói ra Drive (chỉ adapter + log, ~60 MB)
assert P.returncode == 0, "DỪNG: tiến trình train không thoát bằng 0"
ck500 = f"{OUT}/checkpoint-500"
assert du_tep(ck500), "DỪNG: checkpoint-500 thiếu tệp"
L = open(f"{W}/train_{RUN}.log", encoding="utf-8", errors="ignore").read()
print([l for l in L.splitlines() if l.startswith("[xong]")][-1])
print("nan trong log:", bool(re.search(r"\bnan\b", L.lower())))
print("lần nạp default:", [l[:90] for l in L.splitlines() if l.startswith("[nạp default]")])

PK = f"{W}/pack_{RUN}"; shutil.rmtree(PK, ignore_errors=True); os.makedirs(f"{PK}/grpo_spice/checkpoint-500")
for f in ("adapter_model.safetensors", "adapter_config.json"):
    shutil.copy(f"{ck500}/{f}", f"{PK}/grpo_spice/checkpoint-500/{f}")
for f in (f"train_{RUN}.log", "selftest.log", "merge.log"):
    if os.path.exists(f"{W}/{f}"): shutil.copy(f"{W}/{f}", PK)
if os.path.exists(f"{OUT}/log_history.json"): shutil.copy(f"{OUT}/log_history.json", PK)
print("md5 adapter:", md5(f"{PK}/grpo_spice/checkpoint-500/adapter_model.safetensors"))
shutil.make_archive(f"{D}/ck500_{RUN}", "zip", PK)
print("→", f"{D}/ck500_{RUN}.zip", os.path.getsize(f"{D}/ck500_{RUN}.zip") // 2**20, "MB")